# Time and Type of regular days
- weekday vs weekend
- time blocks
- mean relative ranked demand

## data

In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd aus_substation_electricity/

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

In [ ]:
demand.head()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Parse index
demand.index = pd.to_datetime(demand.index)

# 2. Drop columns with too many NaNs (>20% missing)
threshold = 0.2
demand_clean = demand.loc[:, demand.isnull().mean() < threshold]

# 3. Check what got dropped
dropped = demand.columns[demand.isnull().mean() >= threshold].tolist()
print(f"Dropped {len(dropped)} substations: {dropped}")

# 4. Derive time features
demand_clean = demand_clean.copy()
demand_clean['day_of_week'] = demand_clean.index.day_name()
demand_clean['time_block'] = demand_clean.index.strftime('%H:00')

# 5. Normalise each substation to 0–1
substation_cols = [c for c in demand_clean.columns if c not in ['day_of_week', 'time_block']]
demand_norm = demand_clean[substation_cols].apply(lambda x: (x - x.min()) / (x.max() - x.min()))
demand_norm['day_of_week'] = demand_clean['day_of_week']
demand_norm['time_block'] = demand_clean['time_block']

# 6. Pivot
pivot = (
    demand_norm.groupby(['day_of_week', 'time_block'])[substation_cols]
    .mean()
    .mean(axis=1)
    .unstack(level=0)
)

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = pivot[day_order]
pivot = pivot.reindex(sorted(pivot.index))

# 7. Plot
fig, ax = plt.subplots(figsize=(12, 14))
sns.heatmap(
    pivot,
    cmap='YlOrRd',
    ax=ax,
    cbar_kws={'label': 'Mean Normalised Demand'},
    linewidths=0.3,
    linecolor='white'
)
ax.set_title('Demand by Time of Day and Day of Week (all substations)', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('Time of day')
ax.set_yticklabels(ax.get_yticklabels(), rotation=45, ha='right')
ax.tick_params(axis='y', labelsize=10)
plt.tight_layout()
plt.show()

In [ ]:
print(demand_clean['day_of_week'].value_counts())

## Isolating from 7am - 9pm to see finer scale changes

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Parse index
demand.index = pd.to_datetime(demand.index)

# 2. Drop columns with too many NaNs (>20% missing)
threshold = 0.1
demand_clean = demand.loc[:, demand.isnull().mean() < threshold]

# 3. Check what got dropped
dropped = demand.columns[demand.isnull().mean() >= threshold].tolist()
print(f"Dropped {len(dropped)} substations: {dropped}")

# 4. Derive time features
demand_clean = demand_clean.copy()
demand_clean['day_of_week'] = demand_clean.index.day_name()
demand_clean['time_block'] = demand_clean.index.strftime('%H:00')

# 5. Normalise each substation to 0–1
substation_cols = [c for c in demand_clean.columns if c not in ['day_of_week', 'time_block']]
demand_norm = demand_clean[substation_cols].apply(lambda x: (x - x.min()) / (x.max() - x.min()))
demand_norm['day_of_week'] = demand_clean['day_of_week']
demand_norm['time_block'] = demand_clean['time_block']

# 6. Pivot
pivot = (
    demand_norm.groupby(['day_of_week', 'time_block'])[substation_cols]
    .mean()
    .mean(axis=1)
    .unstack(level=0)
)

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = pivot[day_order]
pivot = pivot.reindex(sorted(pivot.index))

# 6.5 Pivot_peak for 7am - 9pm
peak_hours = [f'{h:02d}:00' for h in range(4, 24)]
pivot_peak = pivot.reindex(peak_hours)

# 7. Plot
fig, ax = plt.subplots(figsize=(12, 14))
sns.heatmap(
    pivot_peak,
    cmap='YlOrRd',
    ax=ax,
    cbar_kws={'label': 'Mean Normalised Demand'},
    linewidths=0.3,
    linecolor='white'
)
ax.set_title('Demand by Time of Day and Day of Week (all substations)', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('Time of day')
ax.set_yticklabels(ax.get_yticklabels(), rotation=45, ha='right')
ax.tick_params(axis='y', labelsize=10)
plt.tight_layout()
plt.show()

In [ ]:
info.head()

## Isolating heatmap for substations with residential fraction >0.75

In [ ]:
# Get substations with residential fraction > 0.75
residential_subs = info[info['Residential'] > 0.75].index.tolist()

# Keep only those that survived the NaN threshold drop
residential_subs = [s for s in residential_subs if s in substation_cols]

print(f"{len(residential_subs)} substations with residential fraction > 0.75")

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot_res = (
    demand_norm.groupby(['day_of_week', 'time_block'])[residential_subs]
    .mean()
    .mean(axis=1)
    .unstack(level=0)
)
peak_hours = [f'{h:02d}:00' for h in range(4, 24)]
pivot_res = pivot_res.reindex(peak_hours)[day_order]

# 7. Plot
fig, ax = plt.subplots(figsize=(12, 14))
sns.heatmap(
    pivot_res,
    cmap='YlOrRd',
    ax=ax,
    cbar_kws={'label': 'Mean Normalised Demand'},
    linewidths=0.3,
    linecolor='white'
)
ax.set_title('Demand by Time of Day and Day of Week for Substations with Residential Fraction >0.75', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('Time of day')
ax.set_yticklabels(ax.get_yticklabels(), rotation=45, ha='right')
ax.tick_params(axis='y', labelsize=10)
plt.tight_layout()
plt.show()

# Seeing which substations are making a bigger influence on the demand in the morning peak (higher morning peak in all substations vs res >0.75)

In [ ]:
# Define morning hours
morning = [f'{h:02d}:{m}' for h in range(7, 10) for m in ('00', '30')]

# Non-residential substations (those excluded from residential filter)
non_res_subs = [s for s in substation_cols if s not in residential_subs]

# Mean normalised demand during morning for each substation
morning_mask = demand_norm['time_block'].isin(morning)

# adding weekday filter
weekdays = ['Monday','Tuesday','Wednesday','Thursday','Friday']
morning_mask = demand_norm['time_block'].isin(morning) & demand_norm['day_of_week'].isin(weekdays)

morning_demand = (
    demand_norm[morning_mask]
    .groupby('day_of_week')[substation_cols]
    .mean()
    .mean()
    .sort_values(ascending=False)
)

# Split into res vs non-res
morning_res = morning_demand[morning_demand.index.isin(residential_subs)]
morning_non_res = morning_demand[morning_demand.index.isin(non_res_subs)]

print("Top 10 morning peak substations (residential):")
print(morning_res.head(10))
print("\nTop 10 morning peak substations (non-residential):")
print(morning_non_res.head(10))

In [ ]:
info.loc['KIRRA']

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

morning_res.head(15).plot(kind='barh', ax=axes[0], color='tomato')
axes[0].set_title('Top 15 residential substations\n(weekday morning demand)')
axes[0].set_xlabel('mean normalised demand')
axes[0].invert_yaxis()

morning_non_res.head(15).plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 15 non-residential substations\n(weekday morning demand)')
axes[1].set_xlabel('mean normalised demand')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
info.loc[morning_non_res.head(15).index, ['Name','Commercial', 'Industrial', 'Education','Residential','Persons']]

In [ ]:
info.loc[morning_res.head(15).index, ['Name','Commercial', 'Industrial', 'Education','Residential','Persons']]

In [ ]:
indus_max = max(info['Industrial'])
print(indus_max)

In [ ]:
info[info['Industrial'] == indus_max]

### Evening

In [ ]:
# Define evening hours
evening = [f'{h:02d}:{m}' for h in range(16, 21) for m in ('00', '30')]

# Non-residential substations (those excluded from residential filter)
non_res_subs = [s for s in substation_cols if s not in residential_subs]

# Mean normalised demand during morning for each substation
evening_mask = demand_norm['time_block'].isin(evening)

# adding weekday filter
weekdays = ['Monday','Tuesday','Wednesday','Thursday','Friday']
evening_mask = demand_norm['time_block'].isin(evening) & demand_norm['day_of_week'].isin(weekdays)

evening_demand = (
    demand_norm[evening_mask]
    .groupby('day_of_week')[substation_cols]
    .mean()
    .mean()
    .sort_values(ascending=False)
)

# Split into res vs non-res
evening_res = evening_demand[evening_demand.index.isin(residential_subs)]
evening_non_res = evening_demand[evening_demand.index.isin(non_res_subs)]

print("Top 10 Evening peak substations (residential):")
print(evening_res.head(10))
print("\nTop 10 Evening peak substations (non-residential):")
print(evening_non_res.head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

evening_res.head(15).plot(kind='barh', ax=axes[0], color='tomato')
axes[0].set_title('Top 15 residential substations\n(weekday evening demand)')
axes[0].set_xlabel('mean normalised demand')
axes[0].invert_yaxis()

evening_non_res.head(15).plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 15 non-residential substations\n(weekday evening demand)')
axes[1].set_xlabel('mean normalised demand')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# weekend evening

# Define evening hours
evening = [f'{h:02d}:{m}' for h in range(16, 21) for m in ('00', '30')]

# Non-residential substations (those excluded from residential filter)
non_res_subs = [s for s in substation_cols if s not in residential_subs]

# Mean normalised demand during morning for each substation
evening_mask = demand_norm['time_block'].isin(evening)

# adding weekday filter
weekends = ['Saturday', 'Sunday']
evening_mask = demand_norm['time_block'].isin(evening) & demand_norm['day_of_week'].isin(weekends)

evening_demand = (
    demand_norm[evening_mask]
    .groupby('day_of_week')[substation_cols]
    .mean()
    .mean()
    .sort_values(ascending=False)
)

# Split into res vs non-res
evening_res = evening_demand[evening_demand.index.isin(residential_subs)]
evening_non_res = evening_demand[evening_demand.index.isin(non_res_subs)]

print("Top 10 Evening peak substations (residential):")
print(evening_res.head(10))
print("\nTop 10 Evening peak substations (non-residential):")
print(evening_non_res.head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))
evening_res.head(15).plot(kind='barh', ax=axes[0], color='tomato')
axes[0].set_title('Top 15 residential substations\n(weekend evening demand)')
axes[0].set_xlabel('mean normalised demand')
axes[0].invert_yaxis()

evening_non_res.head(15).plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 15 non-residential substations\n(weekend evening demand)')
axes[1].set_xlabel('mean normalised demand')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# Heat map plotting but with seasons

In [ ]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Summer'
    elif month in [3, 4, 5]:
        return 'Autumn'
    elif month in [6, 7, 8]:
        return 'Winter'
    else:
        return 'Spring'

demand_clean['season'] = demand_clean.index.month.map(get_season)
demand_norm['season'] = demand_clean['season']

In [ ]:
seasons = ['Summer', 'Autumn', 'Winter', 'Spring']
fig, axes = plt.subplots(2, 2, figsize=(14, 16))
axes = axes.flatten()

peak_hours = [f'{h:02d}:00' for h in range(5, 24)]

for i, season in enumerate(seasons):
    season_mask = demand_norm['season'] == season
    
    pivot_season = (
        demand_norm[season_mask]
        .groupby(['day_of_week', 'time_block'])[residential_subs]
        .mean()
        .mean(axis=1)
        .unstack(level=0)
    )
    
    pivot_season = pivot_season.reindex(peak_hours)[day_order]
    
    sns.heatmap(
        pivot_season,
        cmap='YlOrRd',
        ax=axes[i],
        cbar=False,
        linewidths=0.3,
        linecolor='white',
        vmin=0.2,
        vmax=0.55
    )
    axes[i].set_title(f'{season}', fontsize=13)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Time of day')
    axes[i].set_yticklabels(axes[i].get_yticklabels(), rotation=0, ha='right')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='center')
    axes[i].tick_params(axis='y', labelsize=10)

# Universal colourbar
import matplotlib.cm as cm
import matplotlib.colors as mcolors
sm = cm.ScalarMappable(cmap='YlOrRd', norm=mcolors.Normalize(vmin=0.2, vmax=0.55))
sm.set_array([])
cbar_ax = fig.add_axes([0.25, 0.04, 0.5, 0.01])  # [left, bottom, width, height]
fig.colorbar(sm, cax=cbar_ax, label='Mean Normalised Demand', orientation='horizontal')

fig.suptitle('Demand by Time of Day and Day of Week — Residential Substations (>0.75) by Season', 
             fontsize=14, y=0.92)
plt.subplots_adjust(hspace=0.2, wspace=0.3)
# plt.tight_layout()
plt.show()